In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import RandomizedSearchCV, train_test_split, StratifiedKFold, TimeSeriesSplit
from xgboost import XGBClassifier

# Get data on this ticker
def get_data(ticker):
    
    df_orig = yf.Ticker(ticker).history(period="max", interval="1d", auto_adjust=True).reset_index()[['Date', 'Close', 'High', 'Low', 'Volume']]
    df_orig['Date'] = pd.to_datetime(df_orig['Date']).dt.strftime('%Y-%m-%d')
    
    return df_orig

def generate_returns(df, returns):

    def add_column_based_on_future_value(df, days):

        df = df.sort_index(ascending=True)
        future_return = (df['Close'].shift(-days) - df['Close']) / df['Close']
        df[f'Return%_{days}'] = (future_return * 100).round(1) 
        df[f'Return_{days}'] = (future_return > 0).astype(int)
        
        past_return = (df['Close'] - df['Close'].shift(days)) / df['Close']
        df[f'Past_Return_{days}'] = (past_return > 0).astype(int)

        return df

    df_returns = df.copy()

    for r in returns:
    
        df_returns = add_column_based_on_future_value(df_returns, r)

    return df_returns

def ma_features(df): 
    
    df = df_orig.sort_index(ascending=True)

    # =======================
    # Basic SMAs and Ratios
    # =======================
    for window in [5, 10, 25, 50, 100, 200]:
        
        df[f'SMA_{window}'] = df['Close'].rolling(window=window).mean()
        #df[f'EMA_{window}'] = df['Close'].ewm(span=window, adjust=False).mean()

        if window in (50, 100, 200):
            df[f'num_days_{window}'] = 0
    lag_periods = []#[50, 100, 150, 200]
    df = df.sort_index(ascending=False)
    for lag in lag_periods:
        new_cols = {}
        for col in df.columns:
            if col.startswith('SMA_'):
                new_cols[f'{col}_Lag{lag}'] = (df[col] / df[col].shift(-lag)).round(2)

        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
            
    df = df.sort_index(ascending=True)

    for window in []: #[50, 100, 200]:
        for i in range(1, len(df)):
            prev = df.loc[i - 1, f'num_days_{window}']
            price = df.loc[i, 'Close']
            sma = df.loc[i, f'SMA_{window}']
            if price > sma:
                df.loc[i, f'num_days_{window}'] = prev + 1 if prev >= 0 else 0
            elif price < sma:
                df.loc[i, f'num_days_{window}'] = prev - 1 if prev <= 0 else 0
            else:
                df.loc[i, f'num_days_{window}'] = 0

        #df[f'num_days_{window}'] = df[f'num_days_{window}'].apply(lambda x: int(5 * round(x / 5)))

    # ============================
    # Relative Position Features
    # ============================
    def rows_since_max(x): return len(x) - x.argmax() - 1
    def rows_since_min(x): return len(x) - x.argmin() - 1

    for window in [10, 30, 60, 120, 240]:
        df[f'Rel_Max_{window}'] = (df['High'] / df['High'].rolling(window=window).max()).round(2)
        df[f'Rel_Min_{window}'] = (df['Low'] / df['Low'].rolling(window=window).min()).round(2)
        df[f'Max_{window}_Rows_Since'] = df['High'].rolling(window=window).apply(rows_since_max, raw=True)
        df[f'Min_{window}_Rows_Since'] = df['Low'].rolling(window=window).apply(rows_since_min, raw=True)

    for a, b in [(50, 100), (50, 200), (100, 200), (10, 25), (10, 50), (10, 100), (10, 200), (25, 50), (25, 100), (25, 200),
                (5, 10), (5, 25), (5,50)]:    
        df[f'{a}_SMA_{b}'] = (df[f'SMA_{a}'] / df[f'SMA_{b}']).round(2)

    def round_to_nearest_point05(x):
        return round(x * 20) / 20

    for window in [5, 10, 25, 50, 100, 200]:

        df[f'SMA_{window}'] = (df['Close'] / df[f'SMA_{window}']).round(2)
        #df[f'EMA_{window}'] = (df['Close'] / df[f'EMA_{window}']).round(2)

    # RSI
    def RSI(data, period, diff):
        delta = data['Close'].diff(diff)
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        RS = gain / loss
        return (100 - (100 / (1 + RS))).round(0)

    #df['RSI_7'] = RSI(df, 7)
    difs = [1, 3, 5]
    for dif in difs:
        df[f'RSI_14_{dif}'] = RSI(df, 14, dif)
        df[f'RSI_14_{dif}'] = RSI(df, 14, dif)
        df[f'RSI_14_{dif}'] = RSI(df, 14, dif)

    return df

ticker = 'QQQ'
df_orig = get_data(ticker)
df_feats = df_orig.sort_index(ascending=True)

# =======================
# Basic SMAs and Ratios
# =======================
for window in [5, 10, 25, 50, 100, 200]:
    
    df_feats[f'SMA_{window}'] = df_feats['Close'].rolling(window=window).mean()

for window in [5, 10, 20, 50, 100, 200]:

    df_feats[f'Max_{window}'] = (df_feats['Close'] / df_feats['High'].rolling(window=window).max()).round(2)
    df_feats[f'Min_{window}'] = (df_feats['Close'] / df_feats['Low'].rolling(window=window).min()).round(2)

returns = [1, 2, 3, 4, 5, 7, 10, 15, 20, 25, 40, 50]
df_returns = generate_returns(df_orig, returns)
    
df_merged = pd.merge(df_feats, df_returns[[col for col in df_returns.columns if col.startswith('Return')] + ['Date']], on='Date')

# Models

In [3]:
def print_metrics(metrics):
        for thresh, metric_values in metrics.items():
            print(f"  Threshold {thresh}: {metric_values}")
  
def optimize_splits_new(df_indicators, df_predict, thresh, opt, test_size, e, return_metrics=False):
    
    def train_no_eval(model, param_grid, X_train, y_train, X_test, y_test, opt, thresh, iter):

        model.set_params(n_jobs=1)
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            scoring=opt,
            cv=TimeSeriesSplit(n_splits=4),
            n_jobs=-1,
            n_iter=iter,
            random_state=42
        )
        
        #random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        random_search.fit(X_train, y_train, verbose=False)
        best_model = random_search.best_estimator_
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)
            actpos = int(y_test.sum())        # actual positives
            actneg = int(len(y_test) - actpos)  # actual negatives

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 2)
                negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 2)
                predpos = sum(y_pred_valid == 1)
                predneg = sum(y_pred_valid == 0)
                posrec = round(posprec * predpos / postot, 2) if postot else 0
                negrec = round(negprec * predneg / negtot, 2) if negtot else 0
                pos_denom = (2*posrec)+(1*posprec)
                neg_denom = (2*negrec)+(1*negprec)
                posfb = round((3*posprec*posrec)/pos_denom, 2) if pos_denom else 0
                negfb = round((3*negprec*negrec)/neg_denom, 2) if neg_denom else 0
                metrics[t] = {
                    "Eval": 'N',
                    "PAC": actpos,
                    "PPC": predpos,
                    "NAC": actneg,
                    "NPC": predneg,
                    "PP": posprec,
                    "PR": posrec,
                    "NP": negprec,
                    "NR": negrec,
                    "PosFb": posfb,
                    "NegFb": negfb,
                    "PredsRetained": round(((predpos + predneg) / len(y_test)),2)
                }
                metrics[t] = {k: np.nan_to_num(v, nan=0.0) for k, v in metrics[t].items()}
            
        return metrics, best_model
    
    def train_with_eval(model, param_grid, X_train, y_train, X_tune, y_tune, X_test, y_test, opt, thresh, iter):

        model.set_params(n_jobs=1)
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            scoring=opt,
            cv=4,
            n_jobs=-1,
            n_iter=iter,
            random_state=42
        )
        
        random_search.fit(X_train, y_train, eval_set=[(X_tune, y_tune)], verbose=False)
        best_model = random_search.best_estimator_
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)
            actpos = int(y_test.sum())        # actual positives
            actneg = int(len(y_test) - actpos)  # actual negatives

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 2)
                negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 2)
                predpos = sum(y_pred_valid == 1)
                predneg = sum(y_pred_valid == 0)
                posrec = round(posprec * predpos / postot, 2) if postot else 0
                negrec = round(negprec * predneg / negtot, 2) if negtot else 0
                pos_denom = (2*posrec)+(1*posprec)
                neg_denom = (2*negrec)+(1*negprec)
                posfb = round((3*posprec*posrec)/pos_denom, 2) if pos_denom else 0
                negfb = round((3*negprec*negrec)/neg_denom, 2) if neg_denom else 0
                metrics[t] = {
                    "Eval": 'Y',
                    "PAC": actpos,
                    "PPC": predpos,
                    "NAC": actneg,
                    "NPC": predneg,
                    "PP": posprec,
                    "PR": posrec,
                    "NP": negprec,
                    "NR": negrec,
                    "PosFb": posfb,
                    "NegFb": negfb,
                    "PredsRetained": round(((predpos + predneg) / len(y_test)),2)
                }
                metrics[t] = {k: np.nan_to_num(v, nan=0.0) for k, v in metrics[t].items()}
            
        return metrics, best_model

    iter = 50
    
    # Deep
    if e == 'Y': # use eval set

        tune_size = 200 + test_size

        X_test = df_indicators.iloc[:test_size].copy()
        y_test = df_predict.iloc[:test_size].copy()

        X_tune = df_indicators.iloc[test_size:tune_size].copy()
        y_tune = df_predict.iloc[test_size:tune_size].copy()

        X_train = df_indicators.iloc[tune_size:].sort_index(ascending=True).copy()
        y_train = df_predict.iloc[tune_size:].sort_index(ascending=True).copy()

        xgboost_hyperparameters = {
            'max_depth': [3, 6, 9],
            'learning_rate': [0.01, 0.05],    # Lower rates
            'min_child_weight': [6, 10],   # More conservative splits
            'subsample': [0.7, 0.8, 0.9],     
            'colsample_bytree': [0.75, 0.85], 
            'colsample_bylevel': [0.7, 0.8],
            'colsample_bynode': [0.7, 0.8],    
            'gamma': [0.3, 0.5],         
            'alpha': [0.5, 1.0], 
            'lambda': [10, 15],            
            'n_estimators': [200, 400, 600], 
            'early_stopping_rounds': [10]
            }
        
        xg_metrics, best_xg_model = train_with_eval(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, y_train, 
                                                    X_tune, y_tune, X_test, y_test, opt, thresh, iter)
    
    else: # no eval set

        X_test = df_indicators.iloc[:test_size].copy()
        y_test = df_predict.iloc[:test_size].copy()
        
        if withold == 'N':
            test_size = 0
        df_indicators = df_indicators.iloc[test_size:].sort_index(ascending=True).copy()
        df_predict = df_predict.iloc[test_size:].sort_index(ascending=True).copy()

        X_train = df_indicators.copy()
        y_train = df_predict.copy()


        xgboost_hyperparameters = {
            'max_depth': [3, 6, 9],
            'learning_rate': [0.01, 0.05],    # Lower rates
            'min_child_weight': [6, 10],   # More conservative splits
            'subsample': [0.7, 0.8, 0.9],     
            'colsample_bytree': [0.75, 0.85], 
            'colsample_bylevel': [0.7, 0.8],
            'colsample_bynode': [0.7, 0.8],    
            'gamma': [0.3, 0.5],         
            'alpha': [0.5, 1.0], 
            'lambda': [10, 15],            
            'n_estimators': [200, 400, 600], 
            }
        
        xg_metrics, best_xg_model = train_no_eval(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, y_train, 
                                                  X_test, y_test, opt, thresh, iter)
    
    print_metrics(xg_metrics)

    if return_metrics:
        return best_xg_model, xg_metrics
    else:
        #print_metrics(xg_metrics)
        return best_xg_model, best_score

ticker = 'QQQ'
thresh = [.52]
results = []
returns = [5]
arch = 'deep'
withold = 'Y'
test_size = 10
run_perm = "Y"
save_models = 'N'
offsets = [0]
lbs = [8]
td = '10-24'
eval = ['N']
metrics_df = pd.DataFrame()

returns = [15]
#df_orig = get_data(ticker)
#df_feats = ma_features(df_orig)
df_returns = generate_returns(df_orig, returns)
df_merged = pd.merge(df_feats, df_returns[[col for col in df_returns.columns if col.startswith('Return')] + ['Date']], on='Date')

#for test_size in test_sizes:
#for offset in offsets:
for r in returns:

    for i in range(50):

        offset = i * 10

        best_score = -float("inf")
        best_model = None
        best_lb = None
            
        for lb in lbs:

            for e in eval:

                df = df_merged.sort_index(ascending=False).copy()
                cols = [col for col in df.iloc[:, 5:].columns if not col.startswith('Return')]
                record_length = 245 * lb
                df_ph = df.iloc[offset:offset+record_length].copy()
                df_ph = df_ph.iloc[r:].copy()
                return_col = f"Return_{r}"
                return_perc_col = f"Return%_{r}"

                rets = df_ph[return_perc_col].copy()
                # Split into negative and positive returns
                neg = rets[rets < 0]
                pos = rets[rets > 0]
                # Calculate dynamic thresholds
                neg_cutoff = neg.nlargest(int(len(neg) * 0.05)).min()  # least negative of top 10% in magnitude
                pos_cutoff = pos.nsmallest(int(len(pos) * 0.05)).max()  # smallest positive of top 10% in magnitude
                
                filtered = df_ph[(df_ph[f'Return%_{r}'] < neg_cutoff) | (df_ph[f'Return%_{r}'] > pos_cutoff)].copy()
                counts = filtered[return_col].value_counts()
                negf = counts.get(0, 0)
                posf = counts.get(1, 1)  # prevent division by zero
                #print(f'{posf} | {negf}')
                scale_pos_weight = negf / posf

                # Choose evaluation metric
                opt = 'matthews_corrcoef'
                prod_cols = cols

                used_cols = prod_cols + [return_col]
                df_model = filtered[used_cols].dropna()
                
                df_indicators = df_model[prod_cols]
                df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                df_predict = df_model[return_col]
                
                #print(f"Results for {lb}lb | {len(df_ph)}records | Test: {df_ph['Date'].iloc[0]}-{df_ph['Date'].iloc[test_size]} | {ticker}_{r}")
                best_xg_model, metrics = optimize_splits_new(df_indicators, df_predict, thresh, opt, test_size, e, return_metrics=True)

                # convert to DataFrame and include threshold as a column
                df_run = pd.DataFrame.from_dict(metrics, orient='index').reset_index().rename(columns={'index': 'Threshold'})
                df_run['Start_Date'] = df_ph['Date'].iloc[test_size]
                df_run['End_Date'] = df_ph['Date'].iloc[0]
                df_run['LB'] = lb
                df_run['Split'] = 'Temporal'

                # then append to a cumulative DataFrame
                try:
                    metrics_df = pd.concat([metrics_df, df_run], ignore_index=True)
                except NameError:
                    metrics_df = df_run.copy()

  Threshold 0.52: {'Eval': 'N', 'PAC': 10, 'PPC': 9, 'NAC': 0, 'NPC': 0, 'PP': 1.0, 'PR': 0.9, 'NP': 0.0, 'NR': 0, 'PosFb': 0.96, 'NegFb': 0, 'PredsRetained': 0.9}
  Threshold 0.52: {'Eval': 'N', 'PAC': 9, 'PPC': 9, 'NAC': 1, 'NPC': 0, 'PP': 0.89, 'PR': 0.89, 'NP': 0.0, 'NR': 0.0, 'PosFb': 0.89, 'NegFb': 0, 'PredsRetained': 0.9}
  Threshold 0.52: {'Eval': 'N', 'PAC': 10, 'PPC': 5, 'NAC': 0, 'NPC': 4, 'PP': 1.0, 'PR': 0.5, 'NP': 0.0, 'NR': 0, 'PosFb': 0.75, 'NegFb': 0, 'PredsRetained': 0.9}
  Threshold 0.52: {'Eval': 'N', 'PAC': 9, 'PPC': 10, 'NAC': 1, 'NPC': 0, 'PP': 0.9, 'PR': 1.0, 'NP': 0.0, 'NR': 0.0, 'PosFb': 0.93, 'NegFb': 0, 'PredsRetained': 1.0}
  Threshold 0.52: {'Eval': 'N', 'PAC': 5, 'PPC': 8, 'NAC': 5, 'NPC': 2, 'PP': 0.62, 'PR': 0.99, 'NP': 1.0, 'NR': 0.4, 'PosFb': 0.71, 'NegFb': 0.67, 'PredsRetained': 1.0}
  Threshold 0.52: {'Eval': 'N', 'PAC': 9, 'PPC': 10, 'NAC': 1, 'NPC': 0, 'PP': 0.9, 'PR': 1.0, 'NP': 0.0, 'NR': 0.0, 'PosFb': 0.93, 'NegFb': 0, 'PredsRetained': 1.0}
  T